# Vector Data Profiler
Resuable batach data profiler for vector datasets.
Forms stage 1 of data analysis, data understanding.

In [ ]:
# importing libraries
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

In [ ]:
# defining supported formats
SUPPORTED_FORMATS = {
    ".shp",
    ".gpkg",
    ".geojson",
    ".kml"
}

In [17]:
# automaticaly find datasets
# Project root
PROJECT_ROOT = Path.cwd().parent

# Input and output folders
input_folder = PROJECT_ROOT / "data" / "raw"
output_folder = PROJECT_ROOT / "reports"

output_folder.mkdir(exist_ok=True)

# Search dataset folder AND all child folders
datasets = [
    file
    for file in input_folder.rglob("*")
    if file.is_file()
    and file.suffix.lower() in SUPPORTED_FORMATS
]

print(f"Found {len(datasets)} datasets:\n")

for dataset in datasets:
    print(dataset)

    from pathlib import Path

Found 9 datasets:

D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\hotosm_ken_roads_osm\roads_lines.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\hotosm_ken_roads_osm\roads_points.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\hotosm_ken_roads_osm\roads_polygons.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admin0.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admin1.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admin2.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admincentroids.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_adminlines.shp
D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\schools\Schools.shp


In [18]:
# data profiler
def profile_spatial_data(filepath):

    # 1. LOAD DATA
    filepath = Path(filepath)
    gdf = gpd.read_file(filepath)


    # 2. DATASET SUMMARY

    dataset_summary = {
        "file": filepath.name,
        "records": len(gdf),
        "fields": len(gdf.columns),
        "crs": str(gdf.crs),
        "geometry_types":
            gdf.geometry.geom_type.value_counts().to_dict()
    }


    # 3. FIELD PROFILE

    field_profile = []

    for column in gdf.columns:

        series = gdf[column]

        field_type = str(series.dtype)

        missing = series.isna().sum()

        missing_pct = (
            missing / len(gdf) * 100
            if len(gdf) > 0
            else 0
        )

        unique = series.nunique(dropna=True)

        duplicate_values = series.duplicated().sum()

        field_profile.append({
            "Field": column,
            "Type": field_type,
            "Missing": missing,
            "Missing %": round(missing_pct, 2),
            "Unique": unique,
            "Duplicate values": duplicate_values,
        })

    field_profile = pd.DataFrame(field_profile)


    # 4. NUMERIC SUMMARY

    numeric_columns = gdf.select_dtypes(
        include=np.number
    ).columns

    numeric_summary = []

    for column in numeric_columns:

        series = gdf[column]

        numeric_summary.append({
            "Field": column,
            "Min": series.min(),
            "Max": series.max(),
            "Mean": series.mean(),
            "Median": series.median(),
            "Std": series.std(),
            "Zeros": (series == 0).sum(),
            "Negative": (series < 0).sum()
        })

    numeric_summary = pd.DataFrame(numeric_summary)


    # 5. CATEGORICAL SUMMARY

    categorical_summary = {}

    text_columns = gdf.select_dtypes(
        include=["object", "str", "category"]
    ).columns

    for column in text_columns:

        frequency = (
            gdf[column]
            .value_counts(dropna=False)
            .reset_index()
        )

        frequency.columns = [
            column,
            "Count"
        ]

        frequency["Percentage"] = (
            frequency["Count"]
            / len(gdf)
            * 100
        ).round(2)

        categorical_summary[column] = frequency


    # 6. SPATIAL SUMMARY

    spatial_columns = gdf.select_dtypes(
        include="geometry"
    ).columns

    spatial_summary = []

    for column in spatial_columns:

        geometry = gdf[column]
        bounds = geometry.total_bounds

        spatial_summary.append({
            "Field": column,
            "CRS": str(gdf.crs),
            "Geometry types":
                geometry.geom_type.value_counts().to_dict(),
            "Null geometries":
                geometry.isna().sum(),
            "Empty geometries":
                geometry.is_empty.sum(),
            "Invalid geometries":
                (~geometry.is_valid).sum(),
            "Duplicate geometries":
                geometry.duplicated().sum(),
            "Min X": bounds[0],
            "Min Y": bounds[1],
            "Max X": bounds[2],
            "Max Y": bounds[3]
        })

    spatial_summary = pd.DataFrame(spatial_summary)


    # 7. COMPILE REPORT

    report = {
        "dataset": dataset_summary,
        "fields": field_profile,
        "numeric": numeric_summary,
        "categorical": categorical_summary,
        "spatial": spatial_summary
    }

    return report

In [19]:
# write report to excel workbook
def export_spatial_report(report, output_path):

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl"
    ) as writer:

        # 1. Dataset Summary

        dataset_summary = pd.DataFrame(
            list(report["dataset"].items()),
            columns=["Metric", "Value"]
        )

        dataset_summary.to_excel(
            writer,
            sheet_name="Dataset_Summary",
            index=False
        )


        # 2. Field Profile

        report["fields"].to_excel(
            writer,
            sheet_name="Field_Profile",
            index=False
        )


        # 3. Numeric Summary

        report["numeric"].to_excel(
            writer,
            sheet_name="Numeric_Summary",
            index=False
        )


        # 4. Categorical Summary

        categorical_rows = []

        for column, frequency in report["categorical"].items():

            for _, row in frequency.iterrows():

                categorical_rows.append({
                    "Field": column,
                    "Value": row[column],
                    "Count": row["Count"],
                    "Percentage": row["Percentage"]
                })

        categorical_summary = pd.DataFrame(
            categorical_rows
        )

        categorical_summary.to_excel(
            writer,
            sheet_name="Categorical_Summary",
            index=False
        )


        # 5. Spatial Summary

        report["spatial"].to_excel(
            writer,
            sheet_name="Spatial_Summary",
            index=False
        )

In [20]:
# Export file and handle any errors
for filepath in datasets:

    print(f"\nProfiling: {filepath}")

    try:

        report = profile_spatial_data(filepath)

        output_path = (
            output_folder /
            f"{filepath.stem}_dataprofile.xlsx"
        )

        export_spatial_report(
            report,
            output_path
        )

        print(f"✓ Complete: {output_path.name}")

    except Exception as e:

        print(f"✗ Failed: {filepath.name}")
        print(f"  Error: {e}")


Profiling: D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\hotosm_ken_roads_osm\roads_lines.shp
✓ Complete: roads_lines_dataprofile.xlsx

Profiling: D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\hotosm_ken_roads_osm\roads_points.shp
✓ Complete: roads_points_dataprofile.xlsx

Profiling: D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\hotosm_ken_roads_osm\roads_polygons.shp
✓ Complete: roads_polygons_dataprofile.xlsx

Profiling: D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admin0.shp
✓ Complete: ken_admin0_dataprofile.xlsx

Profiling: D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admin1.shp
✓ Complete: ken_admin1_dataprofile.xlsx

Profiling: D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admin2.shp
✓ Complete: ken_admin2_dataprofile.xlsx

Profiling: D:\imma\Geospatial\SpatialSchoolAccessibility\data\raw\ken_admin_boundaries\ken_admincentroids.shp
✓ Complete: ken_a